# Generation Length Distribution Prediction
Predict the **distribution** of token generation lengths (not just the mean) from prompt embeddings.  
Each sample has 128 rollouts → empirical length histogram → soft probability target.  
Models: XGBoost (multi-output), MLP (PyTorch, softmax head).  
Metrics: Wasserstein distance (primary), Jensen-Shannon divergence (secondary).

In [ ]:
# ── Cell 0: Config ────────────────────────────────────────────────────────────
# Identical paths / split as length_regression_pred.ipynb

ROLLOUT_DIR  = "/fs/nexus-scratch/yangfc/cmsc828g-finalproj/rollouts"
EMBED_DIR    = "/fs/nexus-scratch/yangfc/cmsc828g-finalproj/len_pred/preprocess/output"

JSONL_NAME   = "inference_qwen3-4b_math500_G4_B32768_chunk0.jsonl"
EMBED_NAME   = "inference_qwen3-4b_math500_G4_B32768_chunk0__Qwen3-Embedding-4B.parquet"

JSONL_PATH   = f"{ROLLOUT_DIR}/{JSONL_NAME}"
EMBED_PATH   = f"{EMBED_DIR}/{EMBED_NAME}"

# Binning: log-spaced over the observed range.
# 12 bins gives ~10 rollouts/bin on average (128 rollouts / 12 bins).
N_BINS       = 12
BIN_MIN      = 500     # slightly below observed min (~593)
BIN_MAX      = 33000   # slightly above observed max (~32721)

# !! Keep these identical to length_regression_pred.ipynb for a fair comparison !!
SEED         = 42
TEST_SIZE    = 0.2

# MLP training
MLP_EPOCHS      = 100
MLP_BATCH_SIZE  = 32
MLP_LR          = 1e-3

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import json
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import wasserstein_distance
from scipy.special import rel_entr          # for KL / JS
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor

import xgboost as xgb

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# Fix seeds globally (same as length_regression_pred.ipynb)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# ── Cell 1: Define log-spaced bins ────────────────────────────────────────────
bin_edges = np.logspace(np.log10(BIN_MIN), np.log10(BIN_MAX), N_BINS + 1)
# Bin centers (in log space) — used as "positions" for Wasserstein distance
bin_centers = np.sqrt(bin_edges[:-1] * bin_edges[1:])

print(f"Bin edges (rounded):")
print(np.round(bin_edges).astype(int).tolist())
print(f"\nBin centers (rounded):")
print(np.round(bin_centers).astype(int).tolist())

In [ ]:
# ── Cell 2: Build soft label distributions from rollout JSONL ──────────────────
# For each sample_idx, collect all rollout token lengths → empirical histogram → 
# normalize to probability distribution (soft label).

rollout_lengths = defaultdict(list)  # sample_idx -> [token_len, ...]

with open(JSONL_PATH) as f:
    for line in f:
        d = json.loads(line)
        sid = d["sample_idx"]
        for output in d["method_output"]["outputs"]:
            rollout_lengths[sid].append(len(output["token_ids"]))

sample_ids = sorted(rollout_lengths.keys())
n_rollouts_per_sample = len(rollout_lengths[sample_ids[0]])
print(f"Samples: {len(sample_ids)}  |  Rollouts/sample: {n_rollouts_per_sample}")

# Build probability matrix: shape (n_samples, N_BINS)
prob_matrix = np.zeros((len(sample_ids), N_BINS), dtype=np.float32)
for i, sid in enumerate(sample_ids):
    lengths = np.array(rollout_lengths[sid])
    counts, _ = np.histogram(lengths, bins=bin_edges)
    prob_matrix[i] = counts / counts.sum()   # normalize to probability

labels_df = pd.DataFrame(
    prob_matrix,
    columns=[f"bin_{j}" for j in range(N_BINS)]
)
labels_df.insert(0, "sample_idx", sample_ids)

print(f"Label shape: {prob_matrix.shape}  (rows=samples, cols=bins)")
print(f"Row-sum sanity check (should all be 1.0): min={prob_matrix.sum(axis=1).min():.4f}  max={prob_matrix.sum(axis=1).max():.4f}")

In [ ]:
# ── Cell 3: Visualize aggregate distribution & per-sample spread ───────────────
mean_prob = prob_matrix.mean(axis=0)
std_prob  = prob_matrix.std(axis=0)

bin_labels = [f"{int(bin_edges[j])}-{int(bin_edges[j+1])}" for j in range(N_BINS)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Average distribution across all samples
axes[0].bar(range(N_BINS), mean_prob, yerr=std_prob, capsize=3)
axes[0].set_xticks(range(N_BINS))
axes[0].set_xticklabels(bin_labels, rotation=45, ha="right", fontsize=7)
axes[0].set_ylabel("Probability")
axes[0].set_title("Mean (±std) bin probability across samples")

# Heatmap: each row = one sample's length distribution
im = axes[1].imshow(prob_matrix[:100], aspect="auto", cmap="YlOrRd")
axes[1].set_xlabel("Bin")
axes[1].set_ylabel("Sample (first 100)")
axes[1].set_title("Per-sample length distributions (heatmap, first 100 samples)")
plt.colorbar(im, ax=axes[1], label="Probability")

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 4: Load embeddings ────────────────────────────────────────────────────
embed_df = pd.read_parquet(EMBED_PATH)
dim_cols = [c for c in embed_df.columns if c != "sample_idx"]
print(f"Embeddings: {len(embed_df)} samples  |  dim: {len(dim_cols)}")

In [ ]:
# ── Cell 5: Merge & train/test split ──────────────────────────────────────────
# Same SEED and TEST_SIZE as length_regression_pred.ipynb → same sample split.

bin_cols = [f"bin_{j}" for j in range(N_BINS)]
df = embed_df.merge(labels_df, on="sample_idx", how="inner")
print(f"Merged shape: {df.shape}")

X = df[dim_cols].values.astype(np.float32)
Y = df[bin_cols].values.astype(np.float32)  # shape (n_samples, N_BINS)

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=TEST_SIZE, random_state=SEED
)
print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")

In [ ]:
# ── Cell 6: Metrics ────────────────────────────────────────────────────────────
# Wasserstein distance: accounts for ordinal bin structure (distance between
#   bins matters). Best metric when bins are ordered and unequally spaced.
# Jensen-Shannon divergence: symmetric, bounded [0, ln2], always defined
#   (unlike KL which is undefined when predicted prob = 0).

results = {}

def js_divergence(p: np.ndarray, q: np.ndarray, eps: float = 1e-10) -> float:
    """Jensen-Shannon divergence for two probability vectors."""
    m = 0.5 * (p + q) + eps
    return float(0.5 * np.sum(rel_entr(p + eps, m)) + 0.5 * np.sum(rel_entr(q + eps, m)))


def evaluate(name: str, Y_true: np.ndarray, Y_pred_raw: np.ndarray) -> dict:
    """Softmax-normalize predictions then compute per-sample metrics."""
    # Ensure valid probability distributions
    Y_pred = np.exp(Y_pred_raw - Y_pred_raw.max(axis=1, keepdims=True))
    Y_pred = Y_pred / Y_pred.sum(axis=1, keepdims=True)

    ws_distances, js_divs = [], []
    for p_true, p_pred in zip(Y_true, Y_pred):
        ws_distances.append(wasserstein_distance(bin_centers, bin_centers, p_true, p_pred))
        js_divs.append(js_divergence(p_true, p_pred))

    ws  = float(np.mean(ws_distances))
    jsd = float(np.mean(js_divs))
    print(f"{name:20s}  Wasserstein={ws:8.1f}  JSD={jsd:.4f}")
    results[name] = {"Wasserstein": ws, "JSD": jsd, "Y_pred": Y_pred}
    return results[name]

In [ ]:
# ── Cell 7: Baseline — predict mean distribution ───────────────────────────────
# A sensible baseline: predict the training-set mean distribution for every sample.
mean_dist = Y_train.mean(axis=0, keepdims=True)   # shape (1, N_BINS)
Y_pred_baseline = np.repeat(mean_dist, len(Y_test), axis=0)
evaluate("Baseline (mean dist)", Y_test, Y_pred_baseline)

In [ ]:
# ── Cell 8: Model 1 — XGBoost (multi-output) ──────────────────────────────────
# Train one XGBRegressor per bin via MultiOutputRegressor.
# Outputs are softmax-normalized in evaluate().

xgb_base = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    n_jobs=-1,
    verbosity=0,
)
xgb_model = MultiOutputRegressor(xgb_base, n_jobs=1)
xgb_model.fit(X_train, Y_train)

Y_pred_xgb = xgb_model.predict(X_test)  # shape (n_test, N_BINS)
evaluate("XGBoost", Y_test, Y_pred_xgb)

In [ ]:
# ── Cell 9: Model 2 — MLP (PyTorch, softmax output) ───────────────────────────

class MLPDist(nn.Module):
    """MLP with softmax output for probability distribution prediction.
    Extend by changing hidden_dims or output_dim."""
    def __init__(self, input_dim: int, output_dim: int, hidden_dims: list = [512, 128], dropout: float = 0.2):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return F.softmax(self.net(x), dim=-1)


def kl_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """KL(target || pred) — trains the MLP to match the empirical distribution."""
    return (target * (target + eps).log() - target * (pred + eps).log()).sum(dim=-1).mean()


def train_mlp_dist(X_tr, Y_tr, epochs=MLP_EPOCHS, batch_size=MLP_BATCH_SIZE, lr=MLP_LR):
    torch.manual_seed(SEED)
    model = MLPDist(input_dim=X_tr.shape[1], output_dim=N_BINS).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    dataset = TensorDataset(
        torch.from_numpy(X_tr).to(DEVICE),
        torch.from_numpy(Y_tr).to(DEVICE),
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    loss_history = []
    model.train()
    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = kl_loss(pred, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(xb)
        loss_history.append(epoch_loss / len(X_tr))
        if epoch % 20 == 0:
            print(f"  Epoch {epoch:3d}/{epochs}  KL={loss_history[-1]:.4f}")

    return model, loss_history


mlp_model, loss_history = train_mlp_dist(X_train, Y_train)

mlp_model.eval()
with torch.no_grad():
    Y_pred_mlp = mlp_model(torch.from_numpy(X_test).to(DEVICE)).cpu().numpy()
# MLP already outputs softmax probs — pass directly (evaluate() re-normalizes but it's a no-op)
evaluate("MLP", Y_test, Y_pred_mlp)

# Loss curve
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(range(1, MLP_EPOCHS + 1), loss_history)
ax.set_xlabel("Epoch")
ax.set_ylabel("KL divergence (train)")
ax.set_title("MLP training curve")
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 10: Comparison — metrics table + per-sample distribution plots ─────────

# Metrics table
metrics_df = pd.DataFrame(
    {name: {k: v for k, v in vals.items() if k != "Y_pred"} for name, vals in results.items()}
).T
print(metrics_df.to_string(float_format="{:.4f}".format))

# Per-sample Wasserstein distribution (box plots)
ws_per_model = {}
for name, res in results.items():
    Y_pred = res["Y_pred"]
    ws_per_model[name] = [
        wasserstein_distance(bin_centers, bin_centers, Y_test[i], Y_pred[i])
        for i in range(len(Y_test))
    ]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Box plot of per-sample Wasserstein distance
axes[0].boxplot(list(ws_per_model.values()), labels=list(ws_per_model.keys()), patch_artist=True)
axes[0].set_ylabel("Wasserstein distance")
axes[0].set_title("Per-sample Wasserstein (lower = better)")

# Overlay: true vs predicted distribution for a few test samples
sample_indices = [0, 1, 2]   # indices into test set
model_to_plot = [m for m in results if m != "Baseline (mean dist)"][-1]  # last non-baseline model
x = np.arange(N_BINS)
width = 0.35
for k, si in enumerate(sample_indices):
    axes[1].bar(x + k * 0.1, Y_test[si], width=0.3, alpha=0.5, label=f"True #{si}" if k == 0 else None, color="steelblue")
    axes[1].bar(x + k * 0.1 + 0.3, results[model_to_plot]["Y_pred"][si], width=0.3,
                alpha=0.5, label=f"Pred #{si}" if k == 0 else None, color="orange")

axes[1].set_xticks(x)
axes[1].set_xticklabels([f"{int(bin_edges[j])}-" for j in range(N_BINS)], rotation=45, ha="right", fontsize=7)
axes[1].set_ylabel("Probability")
axes[1].set_title(f"True vs Predicted distributions ({model_to_plot}, 3 test samples)")
axes[1].legend(["True", "Predicted"])

plt.tight_layout()
plt.show()